In [0]:
%sql
CREATE OR REPLACE TABLE bank_platform.silver_banking.silver_customers
WITH account as (
SELECT 
    customer_id,
    national_id,
    first_name, 
    last_name,
    cast(date_of_birth as date) as date_of_birth,
    round(datediff(current_date(), cast(date_of_birth as date))/365.25,0) as age,
    email,
    segment,
    risk_rating,
    replace(phone,' ','') as phone,
    address_line,
    city,
    region, 
    postcode,
    kyc_status,
    marketing_consent,
    COALESCE(language, 'English') AS preferred_language,
    cast(updated_at as timestamp) as updated_at,
    cast(created_at as timestamp) as created_at
FROM bank_platform.bronze_banking.bronze_customers as customer
LEFT JOIN 
   read_files("/Volumes/bank_platform/silver_banking/external_data/language.csv") lg
   ON customer.preferred_language = lg.code
   WHERE national_id is not null  order by   round(datediff(current_date(), cast(date_of_birth as date))/365.25,0)
)

select 
*,
       CASE
         WHEN age < 0               THEN 'Invalid'
         WHEN age BETWEEN 0 AND 10  THEN 'Child'
         WHEN age BETWEEN 11 AND 15 THEN 'Youth/Minor_account'
         WHEN age BETWEEN 16 AND 17 THEN 'Young saver'
         WHEN age BETWEEN 18 AND 24 THEN 'Young adult/Student'
         WHEN age BETWEEN 25 AND 59 THEN 'Standard adult'
         WHEN age BETWEEN 60 AND 64 THEN 'Pre-retirement'
         WHEN age >= 65    THEN 'Senior/Pensioner'
         ELSE 'Unknown'
       END AS account_bracket
from account


In [0]:
%sql
select  * from bank_platform.silver_banking.silver_customers

In [0]:
%sql
select * from 
read_files("/Volumes/bank_platform/silver_banking/external_data/language.csv")